# BERT-style MLM pretraining — Phase 1 (WikiText-103) → Phase 2 (UltraFineWeb)

A from-scratch, ~28.7M-param bidirectional encoder trained with masked-language-modeling
on a single 4GB laptop GPU. Two phases:

1. **Phase 1** — pretrain on WikiText-103 (~119M tokens).
2. **Phase 2** — *continued* pretraining, warm-started from phase 1, on ~340M higher-quality
   UltraFineWeb tokens.

Key design choices:
- **Factorized embedding (ALBERT, E=128):** token table is `V×128` then projected `128→768`,
  cutting the embedding from 38.6M → 6.5M params (58% → 22% of the model) so the transformer,
  not a lookup table, dominates — and it fits 4GB.
- **OLMo tokenizer + added `[MASK]`** (OLMo ships none) → vocab 50281.
- **MLM only, no NSP, seq 128** (the budget-BERT recipe).

The heavy lifting lives in importable modules (`encoder.py`, `mlm.py`, `train.py`, `data_prep.py`);
this notebook documents the pipeline and the exact commands.

In [ ]:
import os, sys, torch
ROOT = os.path.abspath("..")           # repo root (notebooks live in notebooks/)
sys.path.insert(0, ROOT)
from types import SimpleNamespace
from mini_enc_transformer import build_tokenizer, build_model
device = "cuda" if torch.cuda.is_available() else "cpu"; device

## Config (fixed: 4 heads, 4 blocks, d_model 768, E=128, seq 128)

In [ ]:
cfg = SimpleNamespace(
    d_model=768, d_embed=128, n_heads=4, n_layers=4, d_k=64, d_v=64, seq_len=128,
    micro_batch=16, grad_accum=16, lr=1e-3, weight_decay=0.01, warmup_frac=0.06,
    max_steps=8800, grad_clip=1.0, mlm_prob=0.15, device=device, seed=0,
)  # micro_batch 16 -> ~1.8GB peak on the 4GB 3050 Ti (measured)

## Tokenizer — OLMo + a `[MASK]` token\nOLMo only ships `<|padding|>`(1) and `<|endoftext|>`(50279); `build_tokenizer` adds `[MASK]` (id 50280) → vocab 50281.

In [ ]:
tk, ids = build_tokenizer("allenai/OLMo-1B-hf")
ids   # {'vocab_size':50281, 'mask_id':50280, 'pad_id':1, 'special_ids':(...)}

## Data — tokenize + pack into a local uint16 memmap

The **network** stage is decoupled from training so a slow/dropped link never interrupts a run.
`data_prep.py` streams (or reads local parquet), applies BERT masking-eligible packing, and writes
a contiguous `uint16` memmap (+ a resumable manifest).

**Phase 1 (WikiText-103), run once in a terminal:**
```bash
python -m mini_enc_transformer.data.prep --dataset Salesforce/wikitext --config wikitext-103-raw-v1 \
    --split train --ufw-shards 0 --shuffle-buffer 0 --min-score 0.0 \
    --target-tokens 120000000 --out data --name wikitext103
```
Packing is contiguous (docs joined by eos) → every 128-token block is full → **no padding, no
attention mask needed** during pretraining.

In [ ]:
# Load the packed blocks (after prep). Masking is applied on the fly each step.
from mini_enc_transformer import PackedMemmapDataset
# train_ds = PackedMemmapDataset("../data", "wikitext103", cfg.seq_len, "train")
# val_ds   = PackedMemmapDataset("../data", "wikitext103", cfg.seq_len, "val")
# len(train_ds), len(val_ds)

## Model — factorized `BertForMaskedLM`

In [ ]:
model = build_model(cfg, ids)
emb = sum(p.numel() for p in model.encoder.embedding.parameters())
tot = sum(p.numel() for p in model.parameters())
print(f"{tot/1e6:.1f}M params | embedding {emb/1e6:.1f}M ({100*emb/tot:.0f}%) | "
      f"decoder tied: {model.mlm_head.decoder.weight is model.encoder.embedding.weight}")

## Phase 1 — train on WikiText-103

`train.py` does bf16 autocast, AdamW, warmup→cosine, grad-accumulation, on-the-fly 80/10/10
masking, atomic checkpoint/resume, and structured metrics for the live dashboard.
```bash
python -m mini_enc_transformer.training.pretrain --data-dir data --data-name wikitext103 \
    --d-model 768 --d-embed 128 --n-heads 4 --n-layers 4 --d-k 64 --d-v 64 --seq-len 128 \
    --micro-batch 16 --grad-accum 16 --lr 1e-3 --warmup-frac 0.06 --max-steps 8800 \
    --ckpt-dir ckpt --device cuda
```
**Result:** step 8800, val_loss 2.544, **masked_acc 55.7%** → `ckpt/last.pt` (the phase-1 backbone).
Budget by STEPS, not wall-clock — a sleeping laptop must not count against the budget.

In [ ]:
# Equivalent in-notebook call (long-running; prefer the CLI above for a real run):
# from mini_enc_transformer import train; train(cfg)

## Phase 2 — continued pretraining on UltraFineWeb (warm-started)

Per scaling laws we were *data-limited*, so the upgrade is better data. UltraFineWeb's 1.3GB
shards don't stream over a slow link, so the shards are downloaded locally to `data/ufw_raw/`
then tokenized offline:
```bash
bash ingest_ufw_local.sh        # tokenizes data/ufw_raw/*.parquet -> data/ultrafineweb_en.bin
bash launch_phase2.sh           # warm-start from ckpt/last.pt, fresh cosine (lr 5e-4) -> ckpt2/
```
`launch_phase2.sh` calls `train.py --init-from ckpt/last.pt` (load **weights only**, fresh
optimizer/schedule/step) so the new data gets its own warmup/cosine. Writes to `ckpt2/` with its
own metrics so the dashboard curves don't collide with phase 1.

In [ ]:
# The warm-start mechanism (weights only, fresh schedule):
#   train.py --init-from ckpt/last.pt  --data-name ultrafineweb_en --ckpt-dir ckpt2 --lr 5e-4 ...
# Phase-2 backbone lands at ckpt2/last.pt, used by the SST-2 fine-tune (see finetune_sst2.ipynb).

## Live dashboard
`python serve_dashboard.py --port 8000` → http://localhost:8000 — loss + masked-accuracy curves,
a data-prep progress panel, and LIVE/DONE status, polling `ckpt*/metrics.jsonl` every 3s.